# Heart Disease Analytics — Exploratory Data Analysis

This notebook documents the exploratory analysis stage of the healthcare analytics project.

**Dataset:** UCI Heart Disease — Cleveland subset  
**Purpose:** understand structure, data quality, distributions, and relationships before predictive modelling.

> This is an analytical/educational project and is not a clinical diagnostic system.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_heart_disease


In [ ]:
df = load_heart_disease()
print(f"Dataset shape: {df.shape}")
display(df.head())


## 1. Data Quality Overview

Check dimensions, data types, missing values, duplicates, and unique values before analysis.


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})
display(quality)
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
display(df.describe(include="all").T)


## 2. Target Distribution

The original UCI target is commonly represented as 0 for no disease and 1–4 for presence of disease. For binary analysis, values greater than 0 are mapped to 1.


In [ ]:
df["target_binary"] = (df["target"] > 0).astype(int)

target_counts = df["target_binary"].value_counts().sort_index()
target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": (target_counts / len(df) * 100).round(2)
})
display(target_summary)

ax = target_counts.rename({0: "No disease", 1: "Disease"}).plot(
    kind="bar", title="Heart Disease Target Distribution"
)
ax.set_xlabel("Target")
ax.set_ylabel("Patients")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 3. Numeric Feature Distributions


In [ ]:
numeric_cols = [
    c for c in ["age", "trestbps", "chol", "thalach", "oldpeak"]
    if c in df.columns
]

for col in numeric_cols:
    df[col].plot(kind="hist", bins=20, title=f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


## 4. Grouped Descriptive Analysis

These statistics compare numerical variables by target. They are descriptive and do not establish causation.


In [ ]:
grouped = (
    df.groupby("target_binary")[numeric_cols]
      .agg(["count", "mean", "median", "std"])
      .round(2)
)
display(grouped)


## 5. Categorical Analysis


In [ ]:
for col in ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        display(pd.crosstab(df[col], df["target_binary"], normalize="columns").round(3))


## 6. Correlation Review

Correlation is an exploratory screening tool, not evidence of clinical causation.


In [ ]:
corr_cols = [c for c in numeric_cols + ["target_binary"] if c in df.columns]
corr = df[corr_cols].corr(numeric_only=True)
display(corr.round(2))

plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlation")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 7. Analytical Decisions

- Preserve raw data separately from transformed data.
- Record missing-value treatment.
- Apply binary target mapping only when required by the modelling question.
- Inspect class balance before selecting evaluation metrics.
- Keep preprocessing inside the modelling pipeline to reduce leakage risk.
- Treat findings as dataset-specific and subject to external validation before clinical use.
